

```
# This is formatted as code
```

# # Phase 1 : PocketPolls Data Format Comparison

## Goal
Compare JSON, Minified JSON, TOON, and YAML for:
- File size
- Token usage
- Readability
- Reverse conversion accuracy
- Suitability for LLM workflows


## Research Question

If the same information is stored in different formats:

- Which format uses the fewest bytes?
- Which format uses the fewest LLM tokens?
- Can the data be converted back without losing information?
- Which format is most suitable for AI workflows?

In [7]:
# ============================================================
# STEP 1: Upload Files
#
# In this step we upload the four versions of the same dataset.
#
# These files contain identical information but are represented
# using different formats.
#
# We will later compare:
# - File size
# - Token count
# - Structural fidelity
#
# Expected files:
#
# sanitized_polls.json
# sanitized_polls.min.json
# sanitized_polls.yaml
# sanitized_polls.toon
# ============================================================

from google.colab import files

uploaded = files.upload()

Saving pocketpolls_JSON_Minified.json to pocketpolls_JSON_Minified (1).json
Saving pocketpolls_JSON_Pretty.json to pocketpolls_JSON_Pretty (1).json
Saving pocketpolls_TOON.toon to pocketpolls_TOON (1).toon
Saving pocketpolls_YAML.yaml to pocketpolls_YAML (1).yaml


In [9]:
# ============================================================
# STEP 2: Verify Files Were Uploaded
#
# Before performing analysis we verify:
#
# 1. All files exist
# 2. Their sizes look reasonable
#
# This helps catch upload issues early.
# ============================================================

import os

print("Uploaded Files")
print("-" * 40)

for filename in uploaded.keys():

    size_bytes = os.path.getsize(filename)
    size_kb = size_bytes / 1024

    print(f"{filename}")
    print(f"Size: {size_kb:.2f} KB")
    print()

Uploaded Files
----------------------------------------
pocketpolls_JSON_Minified (1).json
Size: 115.40 KB

pocketpolls_JSON_Pretty (1).json
Size: 174.19 KB

pocketpolls_TOON (1).toon
Size: 135.93 KB

pocketpolls_YAML (1).yaml
Size: 129.84 KB



In [10]:
# ============================================================
# STEP 3: Read All Files Into Memory
#
# We load every file into a Python dictionary.
#
# Key:
#     filename
#
# Value:
#     entire text content
#
# This allows us to perform size and token analysis later.
# ============================================================

files_data = {}

for filename in uploaded.keys():

    with open(filename, "r", encoding="utf-8") as f:
        files_data[filename] = f.read()

print("Files successfully loaded.")

Files successfully loaded.


In [11]:
# ============================================================
# STEP 4: Preview File Content
#
# We display the first 500 characters of each file.
#
# Purpose:
#
# - Confirm file contents look correct
# - Observe visual differences between formats
#
# This is the first qualitative comparison.
# ============================================================

for filename, content in files_data.items():

    print("=" * 80)
    print(filename)
    print("=" * 80)

    print(content[:250])

    print("\n")

pocketpolls_JSON_Minified (1).json
{"dataset_name":"PocketPolls Vibe Cards — JSON vs TOON Benchmark Dataset","created_for":"Colab case study comparing token efficiency of JSON, minified JSON, and TOON for LLM/RAG prompt context","created_date":"2026-06-13","brand":{"name":"PocketPolls


pocketpolls_JSON_Pretty (1).json
{
  "dataset_name": "PocketPolls Vibe Cards — JSON vs TOON Benchmark Dataset",
  "created_for": "Colab case study comparing token efficiency of JSON, minified JSON, and TOON for LLM/RAG prompt context",
  "created_date": "2026-06-13",
  "brand": {
  


pocketpolls_TOON (1).toon
dataset_name: PocketPolls Vibe Cards — JSON vs TOON Benchmark Dataset
created_for: "Colab case study comparing token efficiency of JSON, minified JSON, and TOON for LLM/RAG prompt context"
created_date: 2026-06-13
brand:
  name: PocketPolls
  tagline


pocketpolls_YAML (1).yaml
dataset_name: PocketPolls Vibe Cards — JSON vs TOON Benchmark Dataset
created_for: Colab case study comparing token effic

In [ ]:
# ============================================================
# STEP 5: Install Tokenizer
#
# LLMs do not process text as characters.
#
# They process text as TOKENS.
#
# A token can be:
#
# - a word
# - part of a word
# - punctuation
# - symbols
#
# Measuring token count helps us estimate:
#
# - LLM cost
# - Context window usage
# - Prompt efficiency
#
# We use OpenAI's tiktoken tokenizer.
# ============================================================

!pip install tiktoken

In [ ]:
# ============================================================
# STEP 6: Token Count Analysis
#
# For each file we calculate:
#
# 1. Storage size
# 2. Number of tokens
#
# This provides our first quantitative comparison.
# ============================================================

import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

results = []

for filename, content in files_data.items():

    size_bytes = len(content.encode("utf-8"))

    token_count = len(
        encoding.encode(content)
    )

    results.append({
        "file": filename,
        "bytes": size_bytes,
        "tokens": token_count
    })

for row in results:

    print("-" * 50)

    print(f"File: {row['file']}")
    print(f"Size (bytes): {row['bytes']:,}")
    print(f"Tokens: {row['tokens']:,}")

--------------------------------------------------
File: pocketpolls_JSON_Minified.json
Size (bytes): 118,167
Tokens: 32,061
--------------------------------------------------
File: pocketpolls_JSON_Pretty.json
Size (bytes): 178,370
Tokens: 42,465
--------------------------------------------------
File: pocketpolls_TOON.toon
Size (bytes): 139,190
Tokens: 35,334
--------------------------------------------------
File: pocketpolls_YAML.yaml
Size (bytes): 132,961
Tokens: 36,615


# Phase 2 - Dataset Fidelity Validation

## Objective

Before drawing conclusions about token efficiency, we must verify that all four formats contain the same information.

A format may appear more efficient simply because information was removed during conversion.

In this phase we will validate:

- Card counts
- Structural completeness
- Metadata preservation

This ensures we are comparing equivalent datasets.

In [ ]:
# ============================================================
# STEP 9
#
# Card Count Validation
#
# Goal:
#
# Verify that all formats contain the same number
# of benchmark cards.
#
# Method:
#
# Count occurrences of "_benchmark_signature".
#
# Expected Result:
#
# All formats should report the same card count.
#
# ============================================================

for filename, content in files_data.items():

    card_count = content.count(
        "_benchmark_signature"
    )

    print(
        f"{filename}: {card_count} cards"
    )

pocketpolls_JSON_Minified.json: 28 cards
pocketpolls_JSON_Pretty.json: 28 cards
pocketpolls_TOON.toon: 28 cards
pocketpolls_YAML.yaml: 28 cards


In [ ]:
# ============================================================
# STEP 10
#
# Structural Validation
#
# Goal:
#
# Verify that major sections exist in all formats.
#
# Sections Checked:
#
# - dataset_name
# - stats
# - cards
# - generation_prompts
#
# ============================================================

for filename, content in files_data.items():

    print("=" * 70)
    print(filename)
    print("=" * 70)

    print(
        "Contains dataset_name:",
        "dataset_name" in content
    )

    print(
        "Contains stats:",
        "stats" in content
    )

    print(
        "Contains cards:",
        "cards" in content
    )

    print(
        "Contains generation_prompts:",
        "generation_prompts" in content
    )

    print()

pocketpolls_JSON_Minified.json
Contains dataset_name: True
Contains stats: True
Contains cards: True
Contains generation_prompts: True

pocketpolls_JSON_Pretty.json
Contains dataset_name: True
Contains stats: True
Contains cards: True
Contains generation_prompts: True

pocketpolls_TOON.toon
Contains dataset_name: True
Contains stats: True
Contains cards: True
Contains generation_prompts: True

pocketpolls_YAML.yaml
Contains dataset_name: True
Contains stats: True
Contains cards: True
Contains generation_prompts: True



In [ ]:
# ============================================================
# STEP 11
#
# Metadata Validation
#
# Goal:
#
# Verify that important benchmark metadata
# appears in all formats.
#
# ============================================================

search_terms = [
    "PocketPolls",
    "AI Vibe Check",
    "2026-06-13"
]

for filename, content in files_data.items():

    print("=" * 70)
    print(filename)

    for term in search_terms:

        print(
            f"{term}:",
            term in content
        )

    print()

pocketpolls_JSON_Minified.json
PocketPolls: True
AI Vibe Check: True
2026-06-13: True

pocketpolls_JSON_Pretty.json
PocketPolls: True
AI Vibe Check: True
2026-06-13: True

pocketpolls_TOON.toon
PocketPolls: True
AI Vibe Check: True
2026-06-13: True

pocketpolls_YAML.yaml
PocketPolls: True
AI Vibe Check: True
2026-06-13: True



In [ ]:
# ============================================================
# STEP 12
#
# Validation Summary
#
# This step provides a quick summary before
# moving to the final analysis phase.
#
# ============================================================

print("Validation Complete")
print()

print("Review the outputs from:")
print("- Step 9 (Card Counts)")
print("- Step 10 (Structure)")
print("- Step 11 (Metadata)")
print()

print(
    "If all values match, the benchmark "
    "can be considered structurally valid."
)

Validation Complete

Review the outputs from:
- Step 9 (Card Counts)
- Step 10 (Structure)
- Step 11 (Metadata)

If all values match, the benchmark can be considered structurally valid.


# Phase 2 Conclusion

The validation process confirmed that all four formats preserved:

- Dataset cardinality (28 cards)
- Major structural sections
- Benchmark metadata

As a result, subsequent comparisons of storage size and token efficiency can be interpreted as comparisons of equivalent datasets rather than comparisons of partially transformed datasets.